## Multi-Modal Input — Image classifier + OCR fallback.

In [1]:
# CELL 1: Install required packages (run once). Remove '!' if running in terminal.

!pip install torch torchvision timm -q
!pip install easyocr -q
!pip install albumentations -q
!pip install opencv-python-headless -q
!pip install pillow -q
print("If packages already installed, skip this. Otherwise run the pip install lines above.")



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Aaditya\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Aaditya\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Aaditya\\AppData\\Local\\Programs\\Python\\

If packages already installed, skip this. Otherwise run the pip install lines above.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!python -m venv alb_env


In [11]:
!alb_env\Scripts\python -m pip install --upgrade pip
!alb_env\Scripts\python -m pip install ipykernel


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
    --------------------------------------- 0.0/1.8 MB 435.7 kB/s eta 0:00:05
   - -------------------------------------- 0.1/1.8 MB 544.7 kB/s eta 0:00:04
   -- ------------------------------------- 0.1/1.8 MB 798.9 kB/s eta 0:00:03
   -- ------------------------------------- 0.1/1.8 MB 798.9 kB/s eta 0:00:03
   --- ------------------------------------ 0.2/1.8 MB 654.6 kB/s eta 0:00:03
   --- ------------------------------------ 0.2/1.8 MB 581.0 kB/s eta 0:00:03
   ---- ----------------------------------- 0.2/1.8 MB 556.2 kB/s eta 0:00:03
   ----- ---------------------------------- 0.2/1.8 MB 597.3 kB/s eta 0:00:03
   ----- ---------------------------------- 0.2/1.8 MB 533.8 kB/s eta 0:00:03
   ----- ---------------------------------- 0.2/1.8 MB 533.8 kB/s eta 0:00:03
   ------- --

In [12]:
!alb_env\Scripts\python -m pip install albumentations==1.4.0


  Using cached albumentations-1.4.0-py3-none-any.whl.metadata (35 kB)
  Using cached scikit_image-0.25.2-cp312-cp312-win_amd64.whl.metadata (14 kB)
  Using cached qudida-0.0.4-py3-none-any.whl.metadata (1.5 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached imageio-2.37.2-py3-none-any.whl.metadata (9.7 kB)
  Using cached tifffile-2025.10.16-py3-none-any.whl.metadata (31 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
Using cached albumentations-1.4.0-py3-none-any.whl (123 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl (12.6 MB)
Using cached qudida-0.0.4-py3-none-any.whl (3.5 kB)
Using cached opencv_python_head

In [13]:
!alb_env\Scripts\python -m ipykernel install --user --name=alb_env --display-name="Albumentations Env"


Installed kernelspec alb_env in C:\Users\Aaditya\AppData\Roaming\jupyter\kernels\alb_env


In [1]:
# ---- INSTALL ALL REQUIRED LIBRARIES INTO alb_env ----

# Base scientific stack
!alb_env\Scripts\python -m pip install --upgrade pip
!alb_env\Scripts\python -m pip install pandas numpy pillow matplotlib tqdm pyyaml

# Computer vision + data loading
!alb_env\Scripts\python -m pip install opencv-python scikit-learn

# PyTorch + torchvision (CUDA 11.8 recommended, works on CPU too)
!alb_env\Scripts\python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Albumentations (with ToTensorV2)
!alb_env\Scripts\python -m pip install albumentations==1.4.0

print("✔ All required packages installed successfully inside alb_env!")


Looking in indexes: https://download.pytorch.org/whl/cu118
✔ All required packages installed successfully inside alb_env!


In [1]:
# ---------- CELL 1: Imports, paths, device, reproducibility ----------
import os, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision

import albumentations as A
from albumentations.pytorch import ToTensorV2

# timm for model backbones
import timm

# Paths (edit if needed)
DATA_DIR = r"C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data"
CSV_FILE = os.path.join(DATA_DIR, "MM_Food_Cleaned_Final.csv")
ART_DIR = os.path.join(DATA_DIR, "artifacts")
IMAGE_BASE = os.path.join(DATA_DIR, "images_processed")
os.makedirs(ART_DIR, exist_ok=True)

print("CSV:", CSV_FILE)
print("IMAGE_BASE:", IMAGE_BASE)
print("ART_DIR:", ART_DIR)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Seeds for reproducibility (best-effort)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

# Speed hint
torch.backends.cudnn.benchmark = True


C:\Users\Aaditya\Desktop\NutriSage_Local\alb_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CSV: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\MM_Food_Cleaned_Final.csv
IMAGE_BASE: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\images_processed
ART_DIR: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts
Device: cuda


In [2]:
# ---------- GPU DIAGNOSTIC CELL ----------
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current CUDA Device:", torch.cuda.current_device())
    print("CUDA Device Name:", torch.cuda.get_device_name(torch.cuda.current_device()))
    print("Total GPU Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    print("Training will run on GPU.")
else:
    print("Training will run on CPU (GPU not detected).")


CUDA Available: True
CUDA Device Count: 1
Current CUDA Device: 0
CUDA Device Name: NVIDIA GeForce RTX 2050
Total GPU Memory (GB): 4.0
Training will run on GPU.


In [3]:
# ---------- CELL 2: Imports and path configuration ----------
import os, json, math, random, time
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch import ToTensorV2

# timm will be used for model
import timm

# Paths (update if needed)
DATA_DIR = r"C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data"
CSV_FILE = os.path.join(DATA_DIR, "MM_Food_Cleaned_Final.csv")
ART_DIR = os.path.join(DATA_DIR, "artifacts")
os.makedirs(ART_DIR, exist_ok=True)

IMAGE_BASE = os.path.join(DATA_DIR, "images_processed")
print("CSV:", CSV_FILE)
print("IMAGE_BASE:", IMAGE_BASE)
print("ART_DIR:", ART_DIR)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == 'cuda':
    try:
        # quick nvidia check if available in the environment
        import subprocess, shlex
        out = subprocess.run(shlex.split("nvidia-smi -L"), capture_output=True, text=True)
        print("nvidia-smi output (first lines):\n", out.stdout.splitlines()[:5])
    except Exception:
        pass

# Reproducibility seeds (optional)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

# cuDNN benchmarking (good for fixed input sizes)
torch.backends.cudnn.benchmark = True


CSV: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\MM_Food_Cleaned_Final.csv
IMAGE_BASE: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\images_processed
ART_DIR: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts
Using device: cuda
nvidia-smi output (first lines):
 ['GPU 0: NVIDIA GeForce RTX 2050 (UUID: GPU-5941fcb8-96b9-c8c6-ce7d-b75e35f4eb9c)']


In [4]:
# ---------- CELL 3: Load CSV and robustly resolve image paths ----------
import ast, re
from pathlib import Path

df = pd.read_csv(CSV_FILE)
if 'mm_id' not in df.columns:
    df = df.reset_index().rename(columns={'index':'mm_id'})

def find_image_path(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return None
    ps = str(p).strip().replace('/', os.sep).replace('\\', os.sep)
    # 1) absolute or direct
    if os.path.isabs(ps) and os.path.exists(ps):
        return ps
    if os.path.exists(ps):
        return os.path.abspath(ps)
    # 2) relative to DATA_DIR
    candidate = os.path.join(DATA_DIR, ps)
    if os.path.exists(candidate):
        return os.path.abspath(candidate)
    # 3) strip up to images_processed if present
    parts = ps.split(os.sep)
    if "images_processed" in parts:
        idx = parts.index("images_processed")
        tail = os.sep.join(parts[idx:])
        candidate2 = os.path.join(DATA_DIR, tail)
        if os.path.exists(candidate2):
            return os.path.abspath(candidate2)
    # 4) try basename in IMAGE_BASE
    try:
        basename = os.path.basename(ps)
        candidate3 = os.path.join(IMAGE_BASE, basename)
        if os.path.exists(candidate3):
            return os.path.abspath(candidate3)
    except Exception:
        pass
    return None

df['image_abs_path'] = df['image_filepath'].apply(find_image_path)

print(f"Total rows: {len(df)}")
found = df['image_abs_path'].apply(lambda x: isinstance(x,str) and os.path.exists(x)).sum()
print(f"Images available (exists): {found}")

# Quick samples
display(df[df['image_abs_path'].notna()]['image_abs_path'].head(10))
display(df[df['image_abs_path'].isna()]['image_filepath'].head(10))


Total rows: 10996
Images available (exists): 10996


0    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
1    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
2    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
3    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
4    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
5    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
6    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
7    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
8    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
9    C:\Users\Aaditya\Desktop\NutriSage_Local\prepr...
Name: image_abs_path, dtype: object

Series([], Name: image_filepath, dtype: object)

In [5]:
# ---------- CELL 4: Label normalization & reduce to TOP_K ----------
LABEL_COL = 'primary_ingredient'   # fallback

import ast, re
from collections import Counter

def extract_primary_ingredient(ing):
    if pd.isna(ing):
        return None
    if isinstance(ing, str):
        try:
            if ing.strip().startswith('['):
                items = ast.literal_eval(ing)
                if isinstance(items, (list,tuple)) and len(items)>0:
                    return str(items[0]).strip().lower()
        except Exception:
            pass
        parts = [p.strip() for p in re.split(r',|\+|;| and | with ', ing) if p.strip()]
        if parts:
            return parts[0].lower()
    return None

# Build primary fields
df['primary_ingredient'] = df.get('primary_ingredient', df['ingredients'].apply(extract_primary_ingredient))
df['dish_label'] = df.get('dish_label', df['dish_name'].astype(str).str.strip().str.lower())

# Normalize raw label text: punctuation removal, lowercasing, simple stemming of plurals
def normalize_label(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > 3 and s.endswith('s'):
        s = s[:-1]
    return s

df['label_raw'] = df.get('primary_ingredient', df.get('dish_label', df.get(LABEL_COL, df.get('ingredients',''))))
df['label_norm'] = df['label_raw'].astype(str).apply(normalize_label)

# Manual mapping for frequent near-duplicates - extend if needed
manual_map = {
    "eggs": "egg",
    "bell peppers": "bell pepper",
    "bell pepper s": "bell pepper",
    "bananas": "banana",
    "apples": "apple",
    "strawberrie": "strawberry",
    "blueberrie": "blueberry",
    "broiler chicken": "chicken"
}
df['label_norm'] = df['label_norm'].replace(manual_map)

print("Top 60 normalized labels (inspect):")
print(df['label_norm'].value_counts().head(60))

# Reduce classes - choose TOP_K (try 30 or smaller experiments)
TOP_K = 30   # default; you can set 15 for a 'hard' run
label_counts = df['label_norm'].value_counts(dropna=True)
top_labels = label_counts.head(TOP_K).index.tolist()
df['label_reduced'] = df['label_norm'].apply(lambda x: x if (isinstance(x,str) and x in top_labels) else 'other')

# Optionally, create a small-experiment dataset with only top N (uncomment when needed)
# TOP_K_SMALL = 12
# top_labels_small = label_counts.head(TOP_K_SMALL).index.tolist()
# df_small = df[df['label_norm'].isin(top_labels_small)].copy()
# df_small['label_reduced'] = df_small['label_norm']

print("Kept top labels:", len(top_labels), "-> sample:", top_labels[:20])
print("Class distribution (top 30):")
print(df['label_reduced'].value_counts().head(30))


Top 60 normalized labels (inspect):
label_norm
beef                755
chicken             602
broth               548
egg                 492
bell pepper         390
meat                277
green onion         274
butter              248
carrot              215
flour               210
bread               200
fish                200
garlic              198
cheese              192
apple               158
cabbage             151
chili               140
broccoli            124
cucumber            115
cilantro            110
noodle              107
fried chicken       102
watermelon          101
banana               99
chicken wing         93
corn                 92
crab                 90
oil                  86
bok choy             85
dumpling wrapper     84
cream                84
chili pepper         82
strawberry           81
bean sprout          75
dough                70
beef patty           69
blueberry            67
lychee               66
pork                 66
crawfish         

In [6]:
# ---------- CELL 5: Split dataset, transforms, dataset class, dataloaders ----------
from sklearn.model_selection import train_test_split
import platform, torch

DEBUG = False   # set True for quick checks

# Use df (reduced labels) by default
df_use = df[df['label_reduced'] != 'other'].copy()   # optionally drop 'other' to focus training
# If you'd rather include 'other', change the line above to df_use = df.copy()

train_df, temp_df = train_test_split(df_use, test_size=0.30, stratify=df_use['label_reduced'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_reduced'], random_state=42)
print("Sizes -> train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

# Image size & batch (safe defaults)
IMG_SIZE = 256
batch_size = 16

# Transforms
if DEBUG:
    train_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])
else:
    train_transform = A.Compose([
        A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.7,1.0), ratio=(0.8,1.2), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.12, rotate_limit=18, p=0.5),
        A.OneOf([
            A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
            A.RandomGamma(gamma_limit=(90,110))
        ], p=0.5),
        A.CoarseDropout(max_holes=1, max_height=int(0.08*IMG_SIZE), max_width=int(0.08*IMG_SIZE), p=0.25),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])

valid_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

class FoodImageDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = class_to_idx
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_abs_path']
        if not isinstance(img_path, str) or not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path} (row idx {idx})")
        img = np.array(Image.open(img_path).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        label = self.class_to_idx.get(row['label_reduced'], self.class_to_idx.get('other', 0))
        return img, label, row['image_abs_path']

# Build placeholder classes mapping now (will be rebuilt in CELL X for exact alignment)
classes = sorted(train_df['label_reduced'].unique())
class_to_idx = {c:i for i,c in enumerate(classes)}
idx_to_class = {i:c for c,i in class_to_idx.items()}

# Create dataset objects
train_ds = FoodImageDataset(train_df, class_to_idx, transform=train_transform)
val_ds = FoodImageDataset(val_df, class_to_idx, transform=valid_transform)
test_ds = FoodImageDataset(test_df, class_to_idx, transform=valid_transform)

_is_windows = platform.system().lower().startswith('win')
num_workers = 0 if _is_windows else 4
pin_memory = True if torch.cuda.is_available() else False

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

print(f"DataLoaders ready. num_workers={num_workers}, pin_memory={pin_memory}.")
x,y,_ = next(iter(train_loader))
print("Example batch shapes:", x.shape, y.shape)


Sizes -> train: 4576 val: 981 test: 981
DataLoaders ready. num_workers=0, pin_memory=True.
Example batch shapes: torch.Size([16, 3, 256, 256]) torch.Size([16])


In [7]:
# ---------- CELL 6: Model (backbone), freeze utils, mixed precision ----------
import timm
from torch import nn
import torch

# Choose backbone
model_name = 'tf_efficientnet_b3_ns'   # good balance b3; change to b4 if you have plenty VRAM/time
print("Model:", model_name)

# We'll create model with num_classes placeholder; CELL X will re-create with exact num_classes later.
try:
    num_classes = len(classes)
except Exception:
    num_classes = 50

model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
model = model.to(device)
print("Created model with num_classes (placeholder):", num_classes)

# Freeze backbone helper
def freeze_backbone(m):
    for name, param in m.named_parameters():
        low = name.lower()
        if not any(x in low for x in ('classifier','fc','head','ln','norm')):
            param.requires_grad = False

def unfreeze_all(m):
    for param in m.parameters():
        param.requires_grad = True

freeze_backbone(model)
print("Backbone frozen (head-only training).")

# Fine-tune prep: unfreeze everything and create a new optimizer for full training
def unfreeze_and_prepare_finetune(model, base_lr=5e-5, weight_decay=1e-4):
    """
    Unfreezes all layers and creates a fresh AdamW optimizer 
    for fine-tuning with lower learning rate.
    """
    # Unfreeze full backbone
    for param in model.parameters():
        param.requires_grad = True

    # New optimizer for full model
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=base_lr,
        weight_decay=weight_decay
    )
    return optimizer


# Use label smoothing by default; actual criterion replaced in CELL X after class weights computed
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Create fast head-only optimizer (recreated in CELL X to ensure correct param groups)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4, weight_decay=1e-4)

# OneCycle flag (recommended)
USE_ONECYCLE = True

# Leave scheduler None; will create after train_loader & class weights in CELL X
scheduler = None

# Mixed precision scaler (if CUDA)
scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

print("Model, optimizer and scaler prepared. USE_ONECYCLE =", USE_ONECYCLE)


Model: tf_efficientnet_b3_ns


C:\Users\Aaditya\Desktop\NutriSage_Local\alb_env\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b3_ns to current tf_efficientnet_b3.ns_jft_in1k.
  model = create_fn(


Created model with num_classes (placeholder): 30
Backbone frozen (head-only training).
Model, optimizer and scaler prepared. USE_ONECYCLE = True


C:\Users\Aaditya\AppData\Local\Temp\ipykernel_13572\11932058.py:66: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None


In [8]:
# ---------- CELL X: Recreate classes, weighted sampler, class weights, and scheduler ----------
import numpy as np
from torch.utils.data.sampler import WeightedRandomSampler
from collections import Counter
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

# Ensure train_df exists
assert 'train_df' in globals(), "Run CELL 5 first so train_df exists."

# Build classes from train split (guarantees mapping alignment)
classes = sorted(train_df['label_reduced'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}   # label->idx
idx_to_class = {i: c for c, i in class_to_idx.items()}  # idx->label
num_classes = len(classes)
print(f"Num classes (train-derived): {num_classes}")

# Recreate model with correct num_classes if needed
if ('model' not in globals()) or (not hasattr(model, 'num_classes')) or (getattr(model, 'num_classes', None) != num_classes):
    print("Recreating model to match num_classes:", num_classes)
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    model = model.to(device)
    try:
        freeze_backbone(model)
    except NameError:
        pass

# class counts
train_label_list = train_df['label_reduced'].tolist()
counts_train = Counter(train_label_list)
class_counts_arr = np.array([counts_train.get(c, 0) for c in classes], dtype=np.float32)
print("Median images per class:", np.median(class_counts_arr))

# Build per-sample weights for WeightedRandomSampler (inverse freq)
fallback_idx = class_to_idx.get('other', 0)
train_labels_idx = [class_to_idx.get(l, fallback_idx) for l in train_df['label_reduced'].tolist()]
weights = [1.0 / (class_counts_arr[idx] if class_counts_arr[idx] > 0 else 1.0) for idx in train_labels_idx]
weights = np.array(weights, dtype=np.float32)
sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

# Recreate train dataset & loader to use sampler (so steps_per_epoch stable)
train_ds = FoodImageDataset(train_df, class_to_idx, transform=train_transform)
# ensure num_workers & pin_memory exist
try:
    num_workers
except NameError:
    num_workers = 0
try:
    pin_memory
except NameError:
    pin_memory = True if torch.cuda.is_available() else False

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=pin_memory)
print("train_loader rebuilt with sampler. Batches per epoch:", len(train_loader))

# Class weights for CrossEntropy (avoid zeros)
class_counts_arr_safe = np.where(class_counts_arr == 0, 1.0, class_counts_arr)
class_weights = 1.0 / class_counts_arr_safe
# normalize class_weights to sum to number of classes (optional)
class_weights = class_weights / class_weights.sum() * len(class_weights)
# pad/truncate to match num_classes (should match)
if len(class_weights) != num_classes:
    padded = np.ones(num_classes, dtype=np.float32)
    padded[:len(class_weights)] = class_weights[:num_classes]
    class_weights = padded
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# Use CrossEntropyLoss with class weights and label smoothing
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
print("Using CrossEntropyLoss with label_smoothing=0.1 and class weights.")

# Recreate optimizer for head-only training (backbone frozen)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4, weight_decay=1e-4)

# Scheduler: OneCycle if requested
if 'num_epochs' not in globals():
    # default if not set yet; will be overwritten by CELL Y when you set epochs
    num_epochs = 30

if 'USE_ONECYCLE' in globals() and USE_ONECYCLE:
    from torch.optim.lr_scheduler import OneCycleLR
    steps_per_epoch = len(train_loader)
    if steps_per_epoch == 0:
        raise RuntimeError("train_loader empty; cannot create OneCycle scheduler.")
    max_lr = 3e-3   # head-only peak (tune if necessary)
    scheduler = OneCycleLR(optimizer, max_lr=max_lr, steps_per_epoch=steps_per_epoch, epochs=num_epochs, pct_start=0.15, anneal_strategy='linear')
    print("OneCycle created: max_lr=", max_lr, "steps_per_epoch=", steps_per_epoch, "epochs=", num_epochs)
else:
    try:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    except TypeError:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    print("Using ReduceLROnPlateau scheduler.")

# Quick checks
print("Example class_to_idx items:", list(class_to_idx.items())[:10])
x_batch, y_batch, _ = next(iter(train_loader))
print("Sample batch shapes after rebuild:", x_batch.shape, y_batch.shape)


Num classes (train-derived): 30
Median images per class: 108.5
train_loader rebuilt with sampler. Batches per epoch: 286
Using CrossEntropyLoss with label_smoothing=0.1 and class weights.
OneCycle created: max_lr= 0.003 steps_per_epoch= 286 epochs= 30
Example class_to_idx items: [('apple', 0), ('banana', 1), ('beef', 2), ('bell pepper', 3), ('bok choy', 4), ('bread', 5), ('broccoli', 6), ('broth', 7), ('butter', 8), ('cabbage', 9)]
Sample batch shapes after rebuild: torch.Size([16, 3, 256, 256]) torch.Size([16])


In [9]:
# ---------- VALIDATION FUNCTION ----------
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += float(loss.item()) * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += int((preds == labels).sum().item())
            total += imgs.size(0)
    val_loss = running_loss / total if total > 0 else float('nan')
    val_acc = correct / total if total > 0 else float('nan')
    return val_loss, val_acc


In [10]:
# ---------- CELL Y: Training driver ----------
import time
from tqdm import tqdm
import numpy as np
import torch

# Training config (tune)
warmup_epochs = 5
finetune_epochs = 45
num_epochs = warmup_epochs + finetune_epochs
grad_clip = 1.0
save_path = os.path.join(ART_DIR, f"{model_name}_food_classifier.pth")
best_val_acc = 0.0
early_stop_patience = 8
no_improve_epochs = 0

# MixUp util
def mixup_data(x, y, alpha=0.4):
    if alpha <= 0:
        return x, y, None, None, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, index, lam

# OneCycle helper to recreate scheduler for finetune stage
def make_onecycle_scheduler(opt, max_lr, epochs, steps):
    from torch.optim.lr_scheduler import OneCycleLR
    return OneCycleLR(opt, max_lr=max_lr, steps_per_epoch=steps, epochs=epochs, pct_start=0.15, anneal_strategy='linear')

print(f"Training plan: {warmup_epochs} warmup + {finetune_epochs} finetune (total {num_epochs})")
print("Saving best model to:", save_path)

# If OneCycle was requested and scheduler is None (edge case), create now
if 'USE_ONECYCLE' in globals() and USE_ONECYCLE and (scheduler is None):
    steps_per_epoch = len(train_loader)
    scheduler = make_onecycle_scheduler(optimizer, max_lr=2e-3, epochs=num_epochs, steps=steps_per_epoch)
    print("Created fallback OneCycle scheduler: max_lr=2e-3")

for epoch in range(1, num_epochs + 1):
    t0 = time.time()
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} (train)", leave=False)
    for imgs, labels, _ in loop:
        imgs = imgs.to(device)
        labels = labels.to(device)

        # MixUp augmentation
        imgs_m, labels_a, labels_b, _, lam = mixup_data(imgs, labels, alpha=0.4)

        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(imgs_m)
                loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(imgs_m)
            loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        # OneCycle per-batch step (ONLY if using OneCycle)
        if ('USE_ONECYCLE' in globals()) and USE_ONECYCLE and (scheduler is not None):
            try:
                scheduler.step()
            except Exception:
                pass

        running_loss += float(loss.item()) * imgs.size(0)
        preds = outputs.argmax(dim=1)
        # training "accuracy" vs original labels (MixUp mixes labels; this is only approximate)
        correct += int((preds == labels).sum().item())
        total += imgs.size(0)
        loop.set_postfix(loss=running_loss/total, acc=correct/total)

    train_loss = running_loss / total if total > 0 else float('nan')
    train_acc = correct / total if total > 0 else float('nan')

    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Scheduler epoch-step (only for ReduceLROnPlateau)
    if not (('USE_ONECYCLE' in globals()) and USE_ONECYCLE) and (scheduler is not None):
        try:
            scheduler.step(val_acc)
        except Exception:
            pass

    if device.type == 'cuda':
        torch.cuda.synchronize()
    t1 = time.time()
    print(f"Epoch {epoch}/{num_epochs} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} | Val Loss {val_loss:.4f} Acc {val_acc:.4f} | time {(t1-t0):.1f}s")

    # checkpoint best
    if val_acc > best_val_acc + 1e-6:
        best_val_acc = val_acc
        no_improve_epochs = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'class_to_idx': class_to_idx,
            'idx_to_class': idx_to_class,
            'model_name': model_name,
            'epoch': epoch
        }, save_path)
        print("Saved best model to", save_path)
    else:
        no_improve_epochs += 1

    # After warmup: unfreeze entire model and recreate optimizer/scheduler for fine-tuning
    if epoch == warmup_epochs and finetune_epochs > 0:
        print("Warmup finished — unfreezing entire model for full fine-tuning.")
        optimizer = unfreeze_and_prepare_finetune(model, base_lr=5e-5)
        if ('USE_ONECYCLE' in globals()) and USE_ONECYCLE:
            steps_per_epoch = len(train_loader)
            # remaining epochs = num_epochs - epoch
            scheduler = make_onecycle_scheduler(optimizer, max_lr=5e-4, epochs=(num_epochs - epoch), steps=steps_per_epoch)
            print("Re-created OneCycle scheduler for fine-tuning: max_lr=5e-4")
        else:
            try:
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
            except TypeError:
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
        print("Re-created optimizer and scheduler for fine-tuning (lower LR).")

    # Early stopping
    if no_improve_epochs >= early_stop_patience:
        print(f"No improvement for {no_improve_epochs} epochs — early stopping.")
        break

print("Training finished. Best validation acc:", best_val_acc)


Training plan: 5 warmup + 45 finetune (total 50)
Saving best model to: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts\tf_efficientnet_b3_ns_food_classifier.pth


Epoch 1/50 (train):   0%|                                                                      | 0/286 [00:00<?, ?it/s]C:\Users\Aaditya\AppData\Local\Temp\ipykernel_13572\110164791.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
                                                                                                                       

Epoch 1/50 | Train Loss 2.9734 Acc 0.1198 | Val Loss 3.1215 Acc 0.1702 | time 51.4s
Saved best model to C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts\tf_efficientnet_b3_ns_food_classifier.pth


Epoch 2/50 | Train Loss 2.5592 Acc 0.1925 | Val Loss 3.2082 Acc 0.2008 | time 49.3s
Saved best model to C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts\tf_efficientnet_b3_ns_food_classifier.pth


Epoch 3/50 | Train Loss 2.5045 Acc 0.2080 | Val Loss 3.1392 Acc 0.2273 | time 48.7s
Saved best model to C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts\tf_efficientnet_b3_ns_food_classifier.pth


...

Epoch 36/50 | Train Loss 1.2863 Acc 0.5275 | Val Loss 2.4879 Acc 0.4608 | time 82.9s


Epoch 37/50 | Train Loss 1.2322 Acc 0.5160 | Val Loss 2.4553 Acc 0.4811 | time 82.6s


Epoch 38/50 | Train Loss 1.3527 Acc 0.4639 | Val Loss 2.4804 Acc 0.4679 | time 82.5s


Epoch 39/50 | Train Loss 1.2955 Acc 0.5219 | Val Loss 2.5075 Acc 0.4628 | time 82.2s


Epoch 40/50 | Train Loss 1.2300 Acc 0.5310 | Val Loss 2.4691 Acc 0.4648 | time 83.2s
No improvement for 8 epochs — early stopping.
Training finished. Best validation acc: 0.4831804281345566


In [11]:
# ---------- CELL Z: Evaluation + diagnostics ----------
from sklearn.metrics import top_k_accuracy_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd
import torch

# Load best checkpoint if exists
if os.path.exists(save_path):
    ckpt = torch.load(save_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    model.eval()
    print("Loaded best model from", save_path, " (epoch:", ckpt.get('epoch', 'N/A'), ")")
    if 'class_to_idx' in ckpt:
        class_to_idx = ckpt['class_to_idx']
        if 'idx_to_class' in ckpt:
            idx_to_class = {int(k): v for k, v in ckpt['idx_to_class'].items()}
        else:
            idx_to_class = {v: k for k, v in class_to_idx.items()}
else:
    print("Warning: best model file not found at", save_path)

def compute_topk_and_preds(loader, model, k=5):
    model.eval()
    y_true = []
    y_pred = []
    probs_all = []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            probs_all.append(probs)
            y_true.extend(labels.numpy().tolist())
            y_pred.extend(preds.tolist())
    if len(probs_all) == 0:
        return None, None, None, None
    probs_all = np.vstack(probs_all)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    top1 = top_k_accuracy_score(y_true, probs_all, k=1)
    top5 = top_k_accuracy_score(y_true, probs_all, k=min(5, probs_all.shape[1]))
    return top1, top5, y_true, y_pred

val_top1, val_top5, y_true_val, y_pred_val = compute_topk_and_preds(val_loader, model, k=5)
print(f"Validation Top-1: {val_top1:.4f}, Top-5: {val_top5:.4f}")

if 'test_loader' in globals():
    test_top1, test_top5, y_true_test, y_pred_test = compute_topk_and_preds(test_loader, model, k=5)
    print(f"Test Top-1: {test_top1:.4f}, Top-5: {test_top5:.4f}")
else:
    print("No test_loader available to compute test metrics.")

# Diagnostics: classification report + confusion matrix for validation
if y_true_val is not None and y_pred_val is not None:
    num_classes = len(idx_to_class)
    label_list = [idx_to_class[i] if i in idx_to_class else str(i) for i in range(num_classes)]
    print("\nClassification report (validation):")
    print(classification_report(y_true_val, y_pred_val, target_names=label_list, zero_division=0))

    cm = confusion_matrix(y_true_val, y_pred_val)
    cm_df = pd.DataFrame(cm, index=label_list, columns=label_list)
    cm_csv = os.path.join(ART_DIR, "val_confusion_matrix.csv")
    cm_df.to_csv(cm_csv)
    print("Saved validation confusion matrix to:", cm_csv)

    diag = np.diag(cm).astype(float)
    row_sums = cm.sum(axis=1).astype(float)
    diag_acc = np.divide(diag, row_sums, out=np.zeros_like(diag), where=row_sums>0)
    worst_idx = np.argsort(diag_acc)[:20]
    print("\nWorst classes by val accuracy (index, class, acc, samples):")
    for i in worst_idx:
        print(i, label_list[i], f"{diag_acc[i]:.3f}", int(row_sums[i]))


Loaded best model from C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts\tf_efficientnet_b3_ns_food_classifier.pth  (epoch: 32 )
Validation Top-1: 0.4832, Top-5: 0.7074
Test Top-1: 0.4567, Top-5: 0.6911

Classification report (validation):
                  precision    recall  f1-score   support

           apple       0.80      0.83      0.82        24
          banana       0.40      0.80      0.53        15
            beef       0.71      0.32      0.45       114
     bell pepper       0.60      0.47      0.52        58
        bok choy       0.37      0.58      0.45        12
           bread       0.44      0.37      0.40        30
        broccoli       0.65      0.61      0.63        18
           broth       0.59      0.63      0.61        82
          butter       0.46      0.51      0.49        37
         cabbage       0.24      0.35      0.28        23
          carrot       0.39      0.38      0.38        32
          cheese       0.56      0.52      0